# 03 Checkpoint verifier restore smoke

Independently verify and restore an exact private checkpoint candidate.

This private `orchestrator_protected` notebook is generated from a reviewed Python template. It receives exact input versions and secrets through Kaggle runtime inputs; no credential is embedded in this notebook or written to its output.

In [ ]:
from __future__ import annotations

import hashlib
import os
import subprocess
import sys
from pathlib import Path

EXPECTED_SOURCE_SHA256 = '3f51629526e62d1fd707a5d5c5b2edb4f82798c05265577c1924affe5e6823a5'
RUNTIME_CONTRACT = 'my-data-hub-checkpoint-restore-smoke.v1'
wheel = Path(os.environ.get('MY_DATA_HUB_WHEEL_PATH', ''))
if not wheel.is_file() or wheel.suffix != '.whl':
    raise RuntimeError('exact private my-data-hub wheel input is required')
expected_wheel_sha = os.environ.get('MY_DATA_HUB_WHEEL_SHA256', '')
if (len(expected_wheel_sha) != 64 or 
        hashlib.sha256(wheel.read_bytes()).hexdigest() != expected_wheel_sha):
    raise RuntimeError('my-data-hub wheel hash mismatch')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-deps', '--disable-pip-version-check', str(wheel)],
    check=True,
)

In [ ]:
PRIMARY_SOURCE = '"""Primary source for an independent checkpoint restore-smoke notebook."""\n\nfrom __future__ import annotations\n\nimport os\nimport shutil\nfrom pathlib import Path\n\nfrom my_data_hub.checkpoints.manifest import load_and_verify\nfrom my_data_hub.checkpoints.verifier import IsolatedPostgresRestoreVerifier\nfrom my_data_hub.hashing import canonical_json_bytes, sha256_file\nfrom my_data_hub.providers.kaggle.adapter import tree_sha256\n\n\ndef _path(name: str) -> Path:\n    value = os.environ.get(name, "").strip()\n    if not value:\n        raise RuntimeError(f"required exact artifact is absent: {name}")\n    path = Path(value)\n    if not path.exists() or path.is_symlink():\n        raise RuntimeError(f"required exact artifact is absent: {name}")\n    return path\n\n\ndef _required(name: str) -> str:\n    value = os.environ.get(name, "").strip()\n    if not value:\n        raise RuntimeError(f"required exact identity is absent: {name}")\n    return value\n\n\ndef main() -> int:\n    package = _path("MY_DATA_HUB_CHECKPOINT_DIRECTORY")\n    manifest_path = _path("MY_DATA_HUB_CHECKPOINT_MANIFEST")\n    if not package.is_dir() or manifest_path.parent.resolve() != package.resolve():\n        raise RuntimeError("checkpoint manifest must be inside the exact private dataset input")\n    expected_package_sha = _required("MY_DATA_HUB_CHECKPOINT_PACKAGE_SHA256")\n    if len(expected_package_sha) != 64 or tree_sha256(package) != expected_package_sha:\n        raise RuntimeError("exact private checkpoint dataset tree hash mismatch")\n    manifest = load_and_verify(manifest_path, package)\n    if str(manifest.checkpoint_id) != _required("MY_DATA_HUB_CHECKPOINT_ID"):\n        raise RuntimeError("checkpoint id differs from the verifier launch identity")\n    if manifest.manifest_sha256 != _required("MY_DATA_HUB_CHECKPOINT_MANIFEST_SHA256"):\n        raise RuntimeError("checkpoint manifest hash differs from the verifier launch identity")\n\n    pg_ctl_value = os.environ.get("MY_DATA_HUB_PG_CTL", "").strip() or shutil.which("pg_ctl") or ""\n    pg_ctl = Path(pg_ctl_value)\n    if not pg_ctl.is_absolute() or not pg_ctl.is_file() or pg_ctl.is_symlink() or not os.access(pg_ctl, os.X_OK):\n        raise RuntimeError("exact executable pg_ctl is required for isolated restore")\n    working = Path("/kaggle/working/checkpoint-restore-work")\n    working.mkdir(mode=0o700, exist_ok=True)\n    port = int(os.environ.get("MY_DATA_HUB_RESTORE_PORT", "55432"))\n    verifier = IsolatedPostgresRestoreVerifier(\n        pg_ctl=pg_ctl,\n        working_directory=working,\n        port=port,\n        timeout_seconds=180,\n    )\n    restore = verifier.verify_restore(package, manifest)\n    observed = {\n        "schema_version": restore["schema_version"],\n        "canonical_revision": restore["canonical_revision"],\n        "logical_hash_sha256": restore["logical_hash_sha256"],\n        "row_counts": restore["row_counts"],\n    }\n    receipt = {\n        "schema_version": "my-data-hub-checkpoint-restore-smoke.v1",\n        "task_run_id": _required("MY_DATA_HUB_VERIFIER_TASK_RUN_ID"),\n        "checkpoint_id": str(manifest.checkpoint_id),\n        "manifest_sha256": manifest.manifest_sha256,\n        "manifest_file_sha256": sha256_file(manifest_path),\n        "dataset_ref": _required("MY_DATA_HUB_CHECKPOINT_DATASET_REF"),\n        "dataset_version": int(_required("MY_DATA_HUB_CHECKPOINT_DATASET_VERSION")),\n        "package_sha256": expected_package_sha,\n        "restore_mode": restore["mode"],\n        "ok": True,\n        "observed": observed,\n    }\n    output = Path("/kaggle/working/checkpoint-restore-receipt.json")\n    output.write_bytes(canonical_json_bytes(receipt))\n    return 0\n'
if hashlib.sha256(PRIMARY_SOURCE.encode()).hexdigest() != EXPECTED_SOURCE_SHA256:
    raise RuntimeError('embedded primary source hash mismatch')
exec(compile(PRIMARY_SOURCE, '<my-data-hub-primary-source>', 'exec'), globals())

In [ ]:
raise SystemExit(globals()['main']())